# Sesión 1 · Demostración guiada

### Comandos elementales de Databricks, listos para ejecutar y explicar


| | |
|---|---|
| Dónde corre | Databricks **Free Edition** (cómputo serverless) |
| Lenguajes | Python y SQL. Scala y R no existen en Free Edition |

**El recorrido:** los magics `%`, crear una tabla, llenarla, modificar los datos y la
estructura, leer el historial, consultar una versión anterior y **restaurarla**.

---
## 1. Los magics: cambiar de lenguaje con `%`

Un notebook tiene **un lenguaje por defecto** —el que eliges al crearlo, aquí Python—
pero no te obliga a quedarte en él. Escribiendo un **magic** en la primera línea,
esa celda (y solo esa) se ejecuta en otro lenguaje.

Las tres celdas que siguen son del mismo notebook y se comportan distinto.

**1.1 · Sin magic.** Se ejecuta en el lenguaje por defecto: Python.

*Verás el texto impreso debajo de la celda.*

In [0]:
mensaje = "Esta celda se ejecutó en Python"
print(mensaje)
print("2 + 2 =", 2 + 2)

**1.2 · Con `%sql`.** La misma celda, pero interpretada como SQL.

*Fíjate en la diferencia: el resultado no sale como texto, sale como una **tabla**
con la que puedes interactuar, ordenar y hasta graficar.*

In [0]:
%sql
SELECT 'Esta celda se ejecutó en SQL' AS mensaje, 2 + 2 AS suma

**1.3 · Con `%md`.** La celda deja de ser código: se convierte en texto con
formato. Es lo que usamos para todas las explicaciones de este notebook.

*La celda siguiente empieza con `%md`. Haz doble clic sobre ella para ver su código
fuente.*

### Esto ya no es código, es documentación

Markdown permite **negritas**, *cursivas*, listas y tablas. Sirve para que un
notebook se lea como un documento y no como un archivo de código suelto.

- Un elemento de lista
- Otro elemento

### Resumen: los magics que importan hoy

| Magic | Qué hace | ¿En Free Edition? |
|---|---|---|
| `%python` | Ejecuta la celda en Python | Sí |
| `%sql` | Ejecuta la celda como SQL | Sí |
| `%md` | Convierte la celda en texto con formato | Sí |
| `%pip` | Instala una librería de Python | Sí, dura lo que la sesión |
| `%run` | Ejecuta otro notebook dentro de este | Sí |
| `%fs` | Atajo para comandos de archivos | Sí, sobre volúmenes |
| `%scala` | Ejecuta la celda en Scala | **No** |
| `%r` | Ejecuta la celda en R | **No** |

**La idea para llevarse:** un solo archivo puede mezclar documentación, Python y SQL.
Por eso el notebook sirve igual para explorar, para documentar y para producir.

---
## 2. Preparar el espacio de trabajo

Antes de crear una tabla hace falta un **esquema** donde guardarla. Recordemos la
dirección de tres niveles de Unity Catalog:

```
workspace  .  practica_databricks  .  ventas
 catálogo         esquema             tabla
```

En Free Edition el catálogo disponible se llama `workspace`. Creamos dentro de él un
esquema propio para no mezclar la demo con nada más.

*Verás un resultado vacío: las sentencias de creación no devuelven filas.*

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.practica_databricks
COMMENT 'Demostración de la Sesión 1'

`USE CATALOG` y `USE SCHEMA` nos **posicionan**, igual que un `cd` en una
carpeta. A partir de aquí ya no haría falta escribir el nombre completo, aunque en
esta demo lo mantendremos para que quede claro de qué tabla hablamos.

In [0]:
%sql
USE CATALOG workspace

In [0]:
%sql
USE SCHEMA practica_databricks

*Esta consulta confirma dónde estamos parados:*

In [0]:
%sql
SELECT current_catalog() AS catalogo, current_schema() AS esquema

---
## 3. Crear una tabla

`CREATE TABLE` define la **estructura**: qué columnas hay y de qué tipo es cada una.

Lo importante es lo que **no** estamos escribiendo: no decimos en qué formato
guardarla. Databricks la crea como **tabla Delta** por defecto, y eso es lo que más
adelante nos permitirá modificarla, versionarla y restaurarla.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.practica_databricks.ventas (
  id        INT            COMMENT 'Identificador de la venta',
  fecha     DATE           COMMENT 'Fecha de la operación',
  ciudad    STRING         COMMENT 'Ciudad donde se vendió',
  producto  STRING         COMMENT 'Producto vendido',
  unidades  INT            COMMENT 'Cantidad de unidades',
  importe   DECIMAL(12,2)  COMMENT 'Importe total en soles'
)
COMMENT 'Tabla de demostración de la Sesión 1'

`SHOW TABLES` lista lo que hay en el esquema.

*Debe aparecer una sola tabla: `ventas`.*

In [0]:
%sql
SHOW TABLES IN workspace.practica_databricks

`DESCRIBE TABLE` muestra su estructura.

*Observa que aparecen las seis columnas con su tipo y el comentario que les pusimos.
Esos comentarios no son decorativos: son los que luego lee el buscador del catálogo
y los que ayudan a Genie a entender la tabla.*

In [0]:
%sql
DESCRIBE TABLE workspace.practica_databricks.ventas

---
## 4. Llenar la tabla

`INSERT INTO` agrega filas. Se pueden mandar varias de una sola vez separándolas por
comas, que es lo normal cuando cargas datos.

*Al ejecutar verás un resumen con el número de filas insertadas.*

In [0]:
%sql
INSERT INTO workspace.practica_databricks.ventas VALUES
  (1, DATE'2026-10-01', 'Lima',     'Laptop',  2, 3200.00),
  (2, DATE'2026-10-01', 'Arequipa', 'Monitor', 1,  950.00),
  (3, DATE'2026-10-02', 'Lima',     'Teclado', 4,  480.00)

Hacemos una segunda carga, por separado. Esto importa para la parte final:
cada `INSERT` genera **una versión distinta** de la tabla, y más adelante vamos a
volver exactamente a este punto.

In [0]:
%sql
INSERT INTO workspace.practica_databricks.ventas VALUES
  (4, DATE'2026-10-03', 'Trujillo', 'Mouse', 5, 225.00)

*Ahora la tabla tiene cuatro filas:*

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.ventas ORDER BY id

---
## 5. Modificar los datos

Aquí está una de las diferencias de fondo entre un Lakehouse y un data lake normal.
Sobre archivos sueltos en la nube **no puedes actualizar ni borrar una fila**:
tendrías que reescribir el archivo entero a mano. Con una tabla Delta, sí.

### 5.1 · `UPDATE`: cambiar valores de las filas que cumplen una condición

*Subimos el importe de la laptop de Lima. El resultado te dirá cuántas filas se
actualizaron.*

In [0]:
%sql
UPDATE workspace.practica_databricks.ventas
SET importe = 3400.00
WHERE ciudad = 'Lima' AND producto = 'Laptop'

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.ventas WHERE ciudad = 'Lima' ORDER BY id

### 5.2 · `DELETE`: borrar filas

*Eliminamos la venta de Mouse. Quedan tres filas.*

In [0]:
%sql
DELETE FROM workspace.practica_databricks.ventas WHERE producto = 'Mouse'

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.ventas ORDER BY id

### 5.3 · `MERGE`: insertar y actualizar en una sola operación

Es el comando más usado en cargas reales. El caso típico: llega un archivo con datos
nuevos y no sabes cuáles ya existen y cuáles no. `MERGE` resuelve los dos casos a la
vez, comparando por una clave.

Primero simulamos los datos que "llegan". La fila `id = 2` **ya existe** en la tabla;
la `id = 5` **es nueva**.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ventas_nuevas AS
SELECT * FROM VALUES
  (2, DATE'2026-10-01', 'Arequipa', 'Monitor', 3, 2850.00),
  (5, DATE'2026-10-04', 'Cusco',    'Laptop',  1, 1600.00)
AS t(id, fecha, ciudad, producto, unidades, importe)

Y ahora el `MERGE`. Léelo en voz alta: *"cuando coincida el id, actualiza;
cuando no coincida, inserta"*.

In [0]:
%sql
MERGE INTO workspace.practica_databricks.ventas AS destino
USING ventas_nuevas AS origen
  ON destino.id = origen.id
WHEN MATCHED     THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.ventas ORDER BY id

*Comprueba el resultado:* la fila `id = 2` cambió sus unidades e importe
(se **actualizó**) y apareció una fila `id = 5` que antes no estaba (se **insertó**).
Una sola instrucción hizo las dos cosas.

---
## 6. Modificar la estructura

Hasta aquí cambiamos **datos**. Ahora cambiamos la **tabla**: le agregamos una
columna que no existía cuando la creamos.

Sobre archivos sueltos esto obliga a reescribirlo todo. En Delta es una operación de
metadatos: `ALTER TABLE` anota la columna nueva en el registro de transacciones y los
archivos de datos que ya estaban **no se tocan**.

*Agregamos `canal`: por dónde entró la venta.*

In [0]:
%sql
ALTER TABLE workspace.practica_databricks.ventas
ADD COLUMNS (canal STRING COMMENT 'Canal por el que entró la venta')

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.ventas ORDER BY id

*Observa la columna nueva:* aparece en **NULL** para todas las filas que ya
existían. Delta no reescribe el pasado, solo amplía la estructura. Ahora la llenamos
con dos actualizaciones.

*La primera marca como venta de tienda todo lo de Lima. La segunda aprovecha
`canal IS NULL` para dejar el resto como `Online`, sin tener que enumerar filas.*

In [0]:
%sql
UPDATE workspace.practica_databricks.ventas
SET canal = 'Tienda'
WHERE ciudad = 'Lima'

In [0]:
%sql
UPDATE workspace.practica_databricks.ventas
SET canal = 'Online'
WHERE canal IS NULL

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.ventas ORDER BY id

*Ya no queda ningún `NULL`.* Con esto la tabla pasó por un `UPDATE`, un `DELETE`,
un `MERGE`, un `ALTER TABLE` y dos actualizaciones más. Todo eso quedó anotado, y es
exactamente lo que vamos a leer ahora.

---
## 7. El historial: todo quedó anotado

Aquí es donde se ve el registro de transacciones del que hablamos en la presentación.
Cada comando que ejecutamos no modificó la tabla "encima" de lo anterior: **escribió
una versión nueva** y dejó constancia.

In [0]:
%sql
DESCRIBE HISTORY workspace.practica_databricks.ventas

Esa salida trae muchas columnas. Las dos que nos interesan ahora:

- **version** — el número que identifica cada estado de la tabla. La `0` es la creación.
- **operation** — qué comando la generó: `CREATE OR REPLACE TABLE`, `WRITE`, `UPDATE`,
  `DELETE`, `MERGE`, `ADD COLUMNS`…

*La celda siguiente deja ese mapa limpio y ordenado de menor a mayor. Es el que
conviene tener a la vista para el resto de la demostración.*

In [0]:
hist = (spark.sql("DESCRIBE HISTORY workspace.practica_databricks.ventas")
             .select("version", "operation")
             .orderBy("version"))
display(hist)

---
## 8. Viajar en el tiempo (sin tocar nada)

`VERSION AS OF` consulta la tabla **como estaba** en un momento dado. Es solo una
lectura: la tabla actual no se modifica.

*Pedimos la versión 1, la que tenía únicamente las tres primeras filas. Fíjate en que
ni siquiera existe la columna `canal`.*

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.ventas VERSION AS OF 1 ORDER BY id

Y se pueden comparar dos momentos en una sola consulta:

In [0]:
%sql
SELECT 'versión 1' AS momento, COUNT(*) AS filas FROM workspace.practica_databricks.ventas VERSION AS OF 1
UNION ALL
SELECT 'ahora',       COUNT(*)             FROM workspace.practica_databricks.ventas

También existe `TIMESTAMP AS OF '2026-10-14T10:00:00'`, útil en el caso real:
casi nunca sabes el número de versión, pero sí recuerdas a qué hora se rompió algo.

---
## 9. Regresar de verdad a una versión anterior

Consultar el pasado es una cosa; **volver** a él es otra. Para eso está `RESTORE`.

Planteemos el caso: *el `DELETE` que hicimos fue un error y queremos deshacerlo*.

*Antes, el estado actual:*

In [0]:
%sql
SELECT COUNT(*) AS filas_ahora FROM workspace.practica_databricks.ventas

Devolvemos la tabla a la **versión 2**: el momento justo después del segundo
`INSERT`, cuando aún no habíamos hecho ni el `UPDATE` ni el `DELETE`.

In [0]:
%sql
RESTORE TABLE workspace.practica_databricks.ventas TO VERSION AS OF 2

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.ventas ORDER BY id

*Mira todo lo que pasó de golpe:* reapareció el Mouse que habíamos borrado, el
importe de la laptop volvió a 3200, se fue la venta de Cusco que había traído el
`MERGE` y desapareció la columna `canal`, que en la versión 2 todavía no existía.

Y lo más importante de todo:

In [0]:
%sql
DESCRIBE HISTORY workspace.practica_databricks.ventas

**El restore también quedó registrado.** La operación más reciente es `RESTORE`.
No borramos el futuro: solo nos movimos en la línea de tiempo, y podríamos volver
hacia adelante usando el número de versión que tenía antes.

Esta es, en una frase, la diferencia entre tener ACID y no tenerlo.

> *Nota técnica:* `RESTORE` es una operación de metadatos, prácticamente instantánea.
> El mismo efecto se consigue con
> `CREATE OR REPLACE TABLE tabla AS SELECT * FROM tabla VERSION AS OF 2`,
> pero eso reescribe todos los datos.

---
## 10. Ir y venir entre SQL y Python

No hay que elegir. El resultado de una consulta SQL **es** un DataFrame de Python, y
un DataFrame se puede exponer a SQL. Es el mismo dato visto desde dos lados.

**De SQL a Python:**

In [0]:
df = spark.sql("SELECT * FROM workspace.practica_databricks.ventas")

print("Filas:", df.count())
df.printSchema()

Y una vez en Python, seguimos trabajando con la API de DataFrames:

In [0]:
from pyspark.sql import functions as F

resumen = (df
    .groupBy("ciudad")
    .agg(F.sum("importe").alias("importe_total"))
    .orderBy(F.desc("importe_total")))

display(resumen)

**De Python a SQL:** una vista temporal expone ese resultado al motor SQL.
Vive solo mientras dure esta sesión del notebook.

In [0]:
resumen.createOrReplaceTempView("resumen_ciudades")
print("Vista temporal 'resumen_ciudades' creada")

In [0]:
%sql
SELECT * FROM resumen_ciudades

Y si queremos que ese resultado deje de ser temporal y pase a ser una tabla de
verdad, guardada y gobernada:

In [0]:
resumen.write.mode("overwrite").saveAsTable("workspace.practica_databricks.resumen_ciudades")
print("Tabla creada: workspace.practica_databricks.resumen_ciudades")

In [0]:
%sql
SELECT * FROM workspace.practica_databricks.resumen_ciudades

---
## 11. Archivos: volúmenes, no DBFS

En Free Edition **DBFS está deshabilitado**, y en el resto de la plataforma está
deprecado. El lugar gobernado para archivos que no son tablas (CSV, PDF, imágenes)
es un **volumen** de Unity Catalog.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.practica_databricks.archivos

Un volumen se comporta como una carpeta normal: se puede escribir en él con
Python puro, sin ninguna librería especial.

In [0]:
with open("/Volumes/workspace/practica_databricks/archivos/demo.csv", "w", encoding="utf-8") as f:
    f.write("ciudad,meta\nLima,5000\nCusco,2000\n")

print("Archivo escrito en /Volumes/workspace/practica_databricks/archivos")

Aquí aparece otro magic: `%fs`, un atajo para listar archivos.

*Verás el CSV que acabamos de crear, con su tamaño.*

In [0]:
%fs ls /Volumes/workspace/practica_databricks/archivos

Lo mismo desde Python, con `dbutils`, la caja de herramientas de Databricks:

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/practica_databricks/archivos"))

Y el archivo se lee como cualquier otra fuente de datos:

In [0]:
df_csv = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/practica_databricks/archivos/demo.csv"))

display(df_csv)

> Si en un tutorial ves rutas como `dbfs:/mnt/...` o `/dbfs/...`, son de la época
> anterior a Unity Catalog. Tradúcelas a `/Volumes/<catálogo>/<esquema>/<volumen>/...`.

---
## 12. `dbutils`: la caja de herramientas del notebook

Ya lo usamos sin presentarlo: `dbutils.fs.ls` en la sección anterior.

`dbutils` es un conjunto de utilidades que Databricks **inyecta en todos los
notebooks**. No se instala y no se importa: simplemente está ahí. Está organizado en
módulos, y cada módulo resuelve una cosa distinta.

| Módulo | Para qué sirve |
|---|---|
| `dbutils.fs` | Archivos: listar, copiar, mover, borrar |
| `dbutils.widgets` | Parámetros de entrada del notebook |
| `dbutils.notebook` | Llamar a otro notebook y recibir un valor |
| `dbutils.secrets` | Usar contraseñas y tokens sin escribirlos |
| `dbutils.jobs` | Pasar valores entre tareas de un job |
| `dbutils.library` | Reiniciar Python tras instalar librerías |
| `dbutils.data` | Resumen estadístico de un DataFrame |
| `dbutils.credentials` | Cambiar de identidad al leer datos |

### 12.1 · `dbutils.help()`: el índice de todo

*El primer comando que conviene recordar, porque te ahorra buscar en la
documentación: lista los módulos disponibles.*

In [0]:
dbutils.help()

Y cada módulo tiene su propia ayuda, con la lista de métodos y lo que recibe cada
uno:

In [0]:
dbutils.fs.help()

### 12.2 · `dbutils.fs`: trabajar con archivos

El módulo más usado. Es el mismo que está detrás del magic `%fs`, pero desde Python
puedes guardar el resultado en una variable y seguir trabajando con él.

*Creamos una carpeta y copiamos dentro el CSV de la sección anterior.*

In [0]:
ruta = "/Volumes/workspace/practica_databricks/archivos"

dbutils.fs.mkdirs(f"{ruta}/respaldo")
dbutils.fs.cp(f"{ruta}/demo.csv", f"{ruta}/respaldo/demo.csv")

display(dbutils.fs.ls(f"{ruta}/respaldo"))

`head` mira el contenido de un archivo **sin** cargarlo en un DataFrame. Es lo
primero que uno hace cuando recibe un archivo y no sabe qué trae dentro:

In [0]:
print(dbutils.fs.head(f"{ruta}/demo.csv"))

Y `rm` lo borra. Ojo con `recurse=True`: borra la carpeta con todo lo que tenga
dentro, sin preguntar.

In [0]:
dbutils.fs.rm(f"{ruta}/respaldo", recurse=True)

display(dbutils.fs.ls(ruta))

### 12.3 · `dbutils.widgets`: convertir el notebook en algo reutilizable

Este es el módulo que cambia la forma de trabajar. Un widget es un **parámetro de
entrada**: en lugar de editar el código para cambiar una ciudad o una fecha, el
notebook te pregunta el valor.

*Al ejecutar la celda siguiente aparecen dos controles arriba del notebook.*

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.dropdown("ciudad", "Lima", ["Lima", "Arequipa", "Trujillo"], "Ciudad")
dbutils.widgets.text("minimo", "0", "Importe mínimo")

print("Widgets creados. Míralos en la parte superior del notebook.")

`get` lee lo que el usuario tenga seleccionado en este momento. Siempre devuelve
**texto**, aunque el widget contenga un número: por eso el `float()`.

In [0]:
ciudad = dbutils.widgets.get("ciudad")
minimo = float(dbutils.widgets.get("minimo"))

print("Ciudad:", ciudad, "| Importe mínimo:", minimo)

Y ese valor entra en la consulta. Fíjate en que **no** pegamos el texto dentro del
SQL: lo pasamos como parámetro con `args`, que es la forma segura de hacerlo.

*Cambia la ciudad en el control de arriba y vuelve a ejecutar esta celda.*

In [0]:
df_filtrado = spark.sql(
    """SELECT *
       FROM workspace.practica_databricks.ventas
       WHERE ciudad = :ciudad AND importe >= :minimo
       ORDER BY id""",
    args={"ciudad": ciudad, "minimo": minimo})

display(df_filtrado)

Desde una celda `%sql` el widget se referencia directamente con `:nombre`, sin pasar
por Python:

In [0]:
%sql
SELECT ciudad, producto, importe
FROM workspace.practica_databricks.ventas
WHERE ciudad = :ciudad
ORDER BY id

*Nota para quien lea tutoriales antiguos:* la sintaxis `${ciudad}` hacía lo mismo y
está deprecada. Usa `:ciudad`.

Y al terminar, los quitamos para dejar el notebook limpio:

In [0]:
dbutils.widgets.removeAll()

print("Widgets eliminados")

---
## 13. ¿Dónde vive físicamente la tabla?

Para cerrar el círculo con lo que vimos en la presentación. `DESCRIBE DETAIL` muestra
la ruta real en el almacenamiento, el formato, cuántos archivos componen la tabla y
cuánto pesa.

In [0]:
%sql
DESCRIBE DETAIL workspace.practica_databricks.ventas

*Las columnas que vale la pena señalar:*

- **format** — dice `delta`.
- **location** — la ruta en el almacenamiento de la nube. Ahí dentro están los
  archivos Parquet y la carpeta `_delta_log`.
- **numFiles** y **sizeInBytes** — cuántos archivos y cuánto ocupan.

Esto es exactamente el diagrama de la presentación, pero con los datos reales de la
tabla que acabamos de construir.

---
## 14. Limpieza

Borra todo lo que creó esta demostración: el esquema, las dos tablas y el volumen.

> Ejecútala al terminar. También conviene ejecutarla **antes** de volver a correr el
> notebook, para que el historial de versiones empiece de cero.

In [0]:
spark.sql("DROP VOLUME IF EXISTS workspace.practica_databricks.archivos")
spark.sql("DROP SCHEMA IF EXISTS workspace.practica_databricks CASCADE")
print("Listo: esquema 'workspace.practica_databricks' eliminado con todo su contenido.")

---
## 15. Lo que acabamos de ver

| Para… | Comando |
|---|---|
| Cambiar el lenguaje de una celda | `%python`, `%sql`, `%md` |
| Crear un esquema | `CREATE SCHEMA IF NOT EXISTS catalogo.esquema` |
| Posicionarse | `USE CATALOG` / `USE SCHEMA` |
| Crear una tabla | `CREATE OR REPLACE TABLE ... ( ... )` |
| Ver su estructura | `DESCRIBE TABLE` |
| Ver dónde vive | `DESCRIBE DETAIL` |
| Listar tablas | `SHOW TABLES IN esquema` |
| Agregar filas | `INSERT INTO ... VALUES ...` |
| Cambiar valores | `UPDATE ... SET ... WHERE ...` |
| Borrar filas | `DELETE FROM ... WHERE ...` |
| Insertar y actualizar a la vez | `MERGE INTO ... USING ... ON ...` |
| Agregar una columna | `ALTER TABLE ... ADD COLUMNS ( ... )` |
| Ver el historial | `DESCRIBE HISTORY` |
| Consultar el pasado | `SELECT ... VERSION AS OF n` |
| **Volver al pasado** | `RESTORE TABLE ... TO VERSION AS OF n` |
| Listar archivos | `%fs ls /Volumes/...` o `dbutils.fs.ls(...)` |
| Copiar o borrar un archivo | `dbutils.fs.cp(...)` / `dbutils.fs.rm(...)` |
| Parametrizar el notebook | `dbutils.widgets.dropdown(...)` + `dbutils.widgets.get(...)` |
| Pedir ayuda sin salir del notebook | `dbutils.help()` / `dbutils.fs.help()` |
| Borrar | `DROP TABLE` / `DROP SCHEMA ... CASCADE` |

